## EN LENGUAJE PYSPARK

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, FloatType, DoubleType, DecimalType, StringType, StructType, StructField
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

In [2]:
# 1. Inicializar la Sesión de Spark
# Asegúrate de tener PySpark instalado y configurado.
spark = SparkSession.builder \
    .appName("Analisis_Clientes_Behavioural") \
    .getOrCreate()


behavioural_df = spark.read.parquet("/home/jovyan/work/data/BEHAVIOURAL", header=True, inferSchema=True)
clientes_df = spark.read.parquet("/home/jovyan/work/data/CLIENTS", header=True, inferSchema=True)

In [3]:
## BEHAVIOURAL

# Reemplaza behavioural_psdf.head() con la función nativa show()
print("Primeras 5 filas de BEHAVIOURAL:")
behavioural_df.show(5, truncate=False)

# Además, añadimos printSchema() para revisar tipos (fundamental en Big Data)
print("Esquema de BEHAVIOURAL:")
behavioural_df.printSchema()

Primeras 5 filas de BEHAVIOURAL:
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|CONTRACT_ID       |CLIENT_ID   |DATE      |CREDICT_CARD_BALANCE|CREDIT_CARD_LIMIT|CREDIT_CARD_DRAWINGS_ATM|CREDIT_CARD_DRAWINGS|CREDIT_CARD_DRAWINGS_POS|CREDIT_CARD_DRAWINGS_OTHER|CREDIT_CARD_PAYMENT|NUMBER_DRAWINGS_ATM|NUMBER_DRAWINGS|NUMBER_INSTALMENTS|CURRENCY|
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|ES1821016961u00XXX|ES182147947X|2020-08-22|0.0                 |2700.0           |0.0                     |0.0                 |0.0                     |0.0            

In [4]:
## CLIENTS

# Reemplaza clients_psdf.head() con la función nativa show()
print("Primeras 5 filas de CLIENTS:")
clientes_df.show(5, truncate=False)

# Además, añadimos printSchema() para revisar tipos (fundamental en Big Data)
print("Esquema de CLIENTS:")
clientes_df.printSchema()

Primeras 5 filas de CLIENTS:
+------------+----------------------+-----------------+------+------------+--------------+-----------+---------------------+--------------+--------------------+------------+------------------+-------------+--------------+-----------+-----------------+-------+-----------+------------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+
|CLIENT_ID   |NON_COMPLIANT_CONTRACT|NAME_PRODUCT_TYPE|GENDER|TOTAL_INCOME|AMOUNT_PRODUCT|INSTALLMENT|EDUCATION            |MARITAL_

In [5]:
# --------- BEHAVIOURAL ---------
# Columnas numéricas
numeric_types = (IntegerType, LongType, FloatType, DoubleType, DecimalType)

behavioural_num_df = behavioural_df.select(
    *[f.name for f in behavioural_df.schema.fields 
      if isinstance(f.dataType, numeric_types)]
)

# Columnas "categóricas" (en Spark normalmente son StringType)
behavioural_cat_df = behavioural_df.select(
    *[f.name for f in behavioural_df.schema.fields 
      if isinstance(f.dataType, StringType)]
)

# --------- CLIENTES ---------
clientes_num_df = clientes_df.select(
    *[f.name for f in clientes_df.schema.fields 
      if isinstance(f.dataType, numeric_types)]
)

clientes_cat_df = clientes_df.select(
    *[f.name for f in clientes_df.schema.fields 
      if isinstance(f.dataType, StringType)]
)

In [6]:
print("Numéricas BEHAVIOURAL:", behavioural_num_df.columns)
print("Categorías BEHAVIOURAL:", behavioural_cat_df.columns)
print("Numéricas CLIENTES:", clientes_num_df.columns)
print("Categorías CLIENTES:", clientes_cat_df.columns)

Numéricas BEHAVIOURAL: ['CREDICT_CARD_BALANCE', 'CREDIT_CARD_LIMIT', 'CREDIT_CARD_DRAWINGS_ATM', 'CREDIT_CARD_DRAWINGS', 'CREDIT_CARD_DRAWINGS_POS', 'CREDIT_CARD_DRAWINGS_OTHER', 'CREDIT_CARD_PAYMENT', 'NUMBER_DRAWINGS_ATM', 'NUMBER_DRAWINGS', 'NUMBER_INSTALMENTS']
Categorías BEHAVIOURAL: ['CONTRACT_ID', 'CLIENT_ID', 'CURRENCY']
Numéricas CLIENTES: ['NON_COMPLIANT_CONTRACT', 'TOTAL_INCOME', 'AMOUNT_PRODUCT', 'INSTALLMENT', 'REGION_SCORE', 'AGE_IN_YEARS', 'JOB_SENIORITY', 'HOME_SENIORITY', 'LAST_UPDATE', 'CAR_AGE', 'FAMILY_SIZE', 'REACTIVE_SCORING', 'PROACTIVE_SCORING', 'BEHAVIORAL_SCORING', 'DAYS_LAST_INFO_CHANGE', 'NUMBER_OF_PRODUCTS', 'DIGITAL_CLIENT', 'NUM_PREVIOUS_LOAN_APP', 'LOAN_ANNUITY_PAYMENT_MAX', 'LOAN_ANNUITY_PAYMENT_MIN', 'LOAN_ANNUITY_PAYMENT_SUM', 'LOAN_APPLICATION_AMOUNT_MAX', 'LOAN_APPLICATION_AMOUNT_MIN', 'LOAN_APPLICATION_AMOUNT_SUM', 'LOAN_CREDIT_GRANTED_MAX', 'LOAN_CREDIT_GRANTED_MIN', 'LOAN_CREDIT_GRANTED_SUM', 'LOAN_VARIABLE_RATE_MAX', 'LOAN_VARIABLE_RATE_MIN', 'N

In [7]:
def calcular_estadisticos_spark(clientes_df):
    
    filas_resumen = []

    for c in clientes_df.columns:

        # MEDIA
        media = clientes_df.select(F.mean(c)).first()[0]

        # MEDIANA
        mediana = clientes_df.approxQuantile(c, [0.5], 0.01)[0]

        # MODA
        moda_row = (
            clientes_df.groupBy(c)
                       .count()
                       .orderBy(F.desc("count"))
                       .first()
        )
        moda = moda_row[0] if moda_row else None

        # MIN & MAX
        stats_minmax = clientes_df.select(F.min(c), F.max(c)).first()
        minimo, maximo = stats_minmax[0], stats_minmax[1]
        rango = maximo - minimo if (minimo is not None and maximo is not None) else None

        # VARIANZA y STD
        stats_var_std = clientes_df.select(F.variance(c), F.stddev(c)).first()
        varianza, desv_std = stats_var_std[0], stats_var_std[1]

        # COEFICIENTE DE VARIACIÓN
        coef_var = desv_std / media if (media not in (None, 0)) else None

        # 👇 Cast a float (double) todo lo numérico
        filas_resumen.append({
            "Columna": str(c),
            "Media": float(media) if media is not None else None,
            "Mediana": float(mediana) if mediana is not None else None,
            "Moda": float(moda) if moda is not None else None,
            "Min": float(minimo) if minimo is not None else None,
            "Max": float(maximo) if maximo is not None else None,
            "Rango": float(rango) if rango is not None else None,
            "Varianza": float(varianza) if varianza is not None else None,
            "DesvStd": float(desv_std) if desv_std is not None else None,
            "CoefVar": float(coef_var) if coef_var is not None else None
        })

    # 📌 Definimos el esquema explícito
    schema = StructType([
        StructField("Columna",  StringType(),  True),
        StructField("Media",    DoubleType(),  True),
        StructField("Mediana",  DoubleType(),  True),
        StructField("Moda",     DoubleType(),  True),
        StructField("Min",      DoubleType(),  True),
        StructField("Max",      DoubleType(),  True),
        StructField("Rango",    DoubleType(),  True),
        StructField("Varianza", DoubleType(),  True),
        StructField("DesvStd",  DoubleType(),  True),
        StructField("CoefVar",  DoubleType(),  True),
    ])

    resumen_clientes = spark.createDataFrame(filas_resumen, schema=schema)

    return resumen_clientes


In [8]:
resumen_clientes = calcular_estadisticos_spark(clientes_num_df)
resumen_clientes.show(truncate=False)

+------------------------+--------------------+------------------+------------------+---------------------+------------------+------------------+---------------------+--------------------+-------------------+
|Columna                 |Media               |Mediana           |Moda              |Min                  |Max               |Rango             |Varianza             |DesvStd             |CoefVar            |
+------------------------+--------------------+------------------+------------------+---------------------+------------------+------------------+---------------------+--------------------+-------------------+
|NON_COMPLIANT_CONTRACT  |0.08121391361971321 |0.0               |0.0               |0.0                  |1.0               |1.0               |0.07461844277751323  |0.273163765491533   |3.363509444583981  |
|TOTAL_INCOME            |2029.3389589942126  |1728.0            |1620.0            |307.8                |1404000.0         |1403692.2         |1.3856957244313696E

In [9]:
resumen_clientes.printSchema()

root
 |-- Columna: string (nullable = true)
 |-- Media: double (nullable = true)
 |-- Mediana: double (nullable = true)
 |-- Moda: double (nullable = true)
 |-- Min: double (nullable = true)
 |-- Max: double (nullable = true)
 |-- Rango: double (nullable = true)
 |-- Varianza: double (nullable = true)
 |-- DesvStd: double (nullable = true)
 |-- CoefVar: double (nullable = true)



In [10]:
# 1️⃣ Filtrar columnas categóricas con ≤ 20 categorías
columnas_validas = []
for c in clientes_cat_df.columns:
    n_cat = clientes_cat_df.select(F.countDistinct(c)).first()[0]
    if n_cat <= 20:
        columnas_validas.append(c)

print("Columnas categóricas con ≤ 20 categorías:")
print(columnas_validas)


Columnas categóricas con ≤ 20 categorías:
['NAME_PRODUCT_TYPE', 'GENDER', 'EDUCATION', 'MARITAL_STATUS', 'HOME_SITUATION', 'OWN_INSURANCE_CAR', 'OCCUPATION', 'HOME_OWNER', 'CURRENCY']


In [11]:
for c in columnas_validas:
    print(f"\n===== Frecuencia de {c} =====")
    frecuencias = (
        clientes_cat_df
        .groupBy(c)
        .count()
        .orderBy(F.desc("count"))
    )
    frecuencias.show(truncate=False)


===== Frecuencia de NAME_PRODUCT_TYPE =====
+-----------------+------+
|NAME_PRODUCT_TYPE|count |
+-----------------+------+
|PRODUCT 1        |294940|
|PRODUCT 2        |31014 |
+-----------------+------+


===== Frecuencia de GENDER =====
+------+------+
|GENDER|count |
+------+------+
|F     |214716|
|M     |111238|
+------+------+


===== Frecuencia de EDUCATION =====
+---------------------+------+
|EDUCATION            |count |
+---------------------+------+
|Secondary            |231648|
|NULL                 |79280 |
|Incomplete University|10814 |
|Primary School       |4034  |
|Master/PhD           |178   |
+---------------------+------+


===== Frecuencia de MARITAL_STATUS =====
+--------------+------+
|MARITAL_STATUS|count |
+--------------+------+
|Married       |240044|
|Single        |85906 |
|NULL          |4     |
+--------------+------+


===== Frecuencia de HOME_SITUATION =====
+-----------------------+------+
|HOME_SITUATION         |count |
+-----------------------+

In [12]:
for col in clientes_num_df.columns:
    print(f"\n===== Histograma de {col} =====")
    
    # Obtener una lista de valores de la columna como RDD
    rdd = clientes_num_df.select(col).rdd.flatMap(lambda x: x)
    
    # Histograma en Spark: 30 bins
    bins, counts = rdd.histogram(30)

    # Mostrar resultado en forma tabular
    hist_df = spark.createDataFrame([
        (float(bins[i]), float(bins[i+1]), int(counts[i]))
        for i in range(len(counts))
    ], ["bin_start", "bin_end", "count"])

    hist_df.show(truncate=False)



===== Histograma de NON_COMPLIANT_CONTRACT =====


+-------------------+-------------------+------+
|bin_start          |bin_end            |count |
+-------------------+-------------------+------+
|0.0                |0.03333333333333333|299482|
|0.03333333333333333|0.06666666666666667|0     |
|0.06666666666666667|0.1                |0     |
|0.1                |0.13333333333333333|0     |
|0.13333333333333333|0.16666666666666666|0     |
|0.16666666666666666|0.2                |0     |
|0.2                |0.23333333333333334|0     |
|0.23333333333333334|0.26666666666666666|0     |
|0.26666666666666666|0.3                |0     |
|0.3                |0.3333333333333333 |0     |
|0.3333333333333333 |0.36666666666666664|0     |
|0.36666666666666664|0.4                |0     |
|0.4                |0.43333333333333335|0     |
|0.43333333333333335|0.4666666666666667 |0     |
|0.4666666666666667 |0.5                |0     |
|0.5                |0.5333333333333333 |0     |
|0.5333333333333333 |0.5666666666666667 |0     |
|0.5666666666666667 

### CORRELACIÓN CLIENTS CON BEHAVIOURAL

In [14]:
# --------- FUNCIÓN GENERAL PARA MATRIZ DE CORRELACIÓN ---------
def compute_corr_matrix(df, df_name="df"):
    """
    Calcula y muestra por pantalla la matriz de correlación de Pearson
    de un DataFrame con solo columnas numéricas, eliminando filas con nulls.
    """
    numeric_cols = df.columns
    if len(numeric_cols) == 0:
        print(f"No hay columnas numéricas en {df_name}")
        return

    # 1) Eliminamos filas que tengan null en alguna de las columnas numéricas
    df_no_nulls = df.na.drop(subset=numeric_cols)

    # 2) Montamos el vector de características
    assembler = VectorAssembler(
        inputCols=numeric_cols,
        outputCol="features"  # handleInvalid por defecto es "error", pero ya no hay nulls
    )
    vector_df = assembler.transform(df_no_nulls).select("features")

    # 3) Correlación de Pearson
    corr_matrix = Correlation.corr(vector_df, "features", "pearson").head()[0]
    corr_array = corr_matrix.toArray().tolist()  # lista de listas de Python

    # Impresión (opcional)
    print(f"\nMatriz de correlación para {df_name}:")
    header = " " * 15 + " ".join([f"{c:>12}" for c in numeric_cols])
    print(header)
    for col_name, row in zip(numeric_cols, corr_array):
        row_str = " ".join([f"{v:12.4f}" for v in row])
        print(f"{col_name:>15} {row_str}")

    return numeric_cols, corr_array

# --------- CÁLCULO PARA CADA DATASET ---------
beh_cols, beh_corr = compute_corr_matrix(behavioural_num_df, "behavioural_num_df")
cli_cols, cli_corr = compute_corr_matrix(clientes_num_df, "clientes_num_df")


Matriz de correlación para behavioural_num_df:
               CREDICT_CARD_BALANCE CREDIT_CARD_LIMIT CREDIT_CARD_DRAWINGS_ATM CREDIT_CARD_DRAWINGS CREDIT_CARD_DRAWINGS_POS CREDIT_CARD_DRAWINGS_OTHER CREDIT_CARD_PAYMENT NUMBER_DRAWINGS_ATM NUMBER_DRAWINGS NUMBER_INSTALMENTS
CREDICT_CARD_BALANCE       1.0000       0.5029       0.2977       0.3381       0.1800       0.0688       0.1704       0.3304       0.2600       0.0360
CREDIT_CARD_LIMIT       0.5029       1.0000       0.2043       0.2707       0.1984       0.0423       0.2434       0.1815       0.2127      -0.1280
CREDIT_CARD_DRAWINGS_ATM       0.2977       0.2043       1.0000       0.8136       0.0889       0.0191       0.1798       0.7354       0.3082      -0.0711
CREDIT_CARD_DRAWINGS       0.3381       0.2707       0.8136       1.0000       0.6028       0.2415       0.3040       0.6074       0.5269      -0.0949
CREDIT_CARD_DRAWINGS_POS       0.1800       0.1984       0.0889       0.6028       1.0000       0.0094       0.2946     

UNIÓN DE LOS DATASETS

In [16]:
# Unión de datasets (inner, puedes usar left o full si quieres)
combined_df = behavioural_df.join(clientes_df, on="CLIENT_ID", how="inner")

In [19]:
combined_num_df = combined_df.select(
    *[f.name for f in combined_df.schema.fields if isinstance(f.dataType, numeric_types)]
)

In [25]:
def compute_corr_matrix(df, df_name="df"):
    numeric_cols = df.columns
    if len(numeric_cols) == 0:
        print(f"No hay columnas numéricas en {df_name}")
        return

    # Eliminar filas con nulls
    df_no_nulls = df.na.drop(subset=numeric_cols)

    # Ensamblar vector
    assembler = VectorAssembler(
        inputCols=numeric_cols,
        outputCol="features"
    )
    vector_df = assembler.transform(df_no_nulls).select("features")

    # Correlación Pearson
    corr_matrix = Correlation.corr(vector_df, "features", "pearson").head()[0]
    corr_array = corr_matrix.toArray().tolist()

    print(f"\nMatriz de correlación para {df_name}:")
    header = " " * 15 + " ".join([f"{c:>12}" for c in numeric_cols])
    print(header)
    for col_name, row in zip(numeric_cols, corr_array):
        row_str = " ".join([f"{v:12.4f}" for v in row])
        print(f"{col_name:>15} {row_str}")

    return numeric_cols, corr_array



In [26]:
combined_cols, combined_corr = compute_corr_matrix(
    combined_num_df,
    "combined_df"
)


Matriz de correlación para combined_df:
               CREDICT_CARD_BALANCE CREDIT_CARD_LIMIT CREDIT_CARD_DRAWINGS_ATM CREDIT_CARD_DRAWINGS CREDIT_CARD_DRAWINGS_POS CREDIT_CARD_DRAWINGS_OTHER CREDIT_CARD_PAYMENT NUMBER_DRAWINGS_ATM NUMBER_DRAWINGS NUMBER_INSTALMENTS NON_COMPLIANT_CONTRACT TOTAL_INCOME AMOUNT_PRODUCT  INSTALLMENT REGION_SCORE AGE_IN_YEARS JOB_SENIORITY HOME_SENIORITY  LAST_UPDATE      CAR_AGE  FAMILY_SIZE REACTIVE_SCORING PROACTIVE_SCORING BEHAVIORAL_SCORING DAYS_LAST_INFO_CHANGE NUMBER_OF_PRODUCTS DIGITAL_CLIENT NUM_PREVIOUS_LOAN_APP LOAN_ANNUITY_PAYMENT_MAX LOAN_ANNUITY_PAYMENT_MIN LOAN_ANNUITY_PAYMENT_SUM LOAN_APPLICATION_AMOUNT_MAX LOAN_APPLICATION_AMOUNT_MIN LOAN_APPLICATION_AMOUNT_SUM LOAN_CREDIT_GRANTED_MAX LOAN_CREDIT_GRANTED_MIN LOAN_CREDIT_GRANTED_SUM LOAN_VARIABLE_RATE_MAX LOAN_VARIABLE_RATE_MIN NUM_STATUS_ANNULLED NUM_STATUS_AUTHORIZED NUM_STATUS_DENIED NUM_STATUS_NOT_USED NUM_FLAG_INSURED
CREDICT_CARD_BALANCE       1.0000       0.5251       0.2923       0.

Al pasarlo a PySpark hay un error del orden de 0.00x a 0.02–0.03 en muchos coeficientes.
Ejemplo que tú misma has puesto:

Spark: 0.5251

Python: 0.502851
→ diferencia ≈ 0.022

Eso no es enorme en términos de correlación:

- No cambia el signo (positiva sigue siendo positiva, negativa sigue siendo negativa).

- La interpretación “fuerte / débil / casi 0” suele ser la misma.

- Pero si estás comparando variables muy similares entre sí (por ejemplo para ordenar features por importancia) esas pequeñas diferencias sí podrían cambiar el ranking entre dos variables muy parecidas.